In [2]:
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"

client = bigquery.Client(
    project=PROJECT_ID,
    location="asia-northeast3"
)

print("연결 준비 완료")

연결 준비 완료


/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [3]:
import pandas as pd

table_names = [
    "accounts_userquestionrecord",
    "polls_questionset",
    "hackle_properties",
]

schema_rows = []

for table_name in table_names:
    table_id = f"{PROJECT_ID}.sns_analysis.{table_name}"
    table = client.get_table(table_id)

    for field in table.schema:
        schema_rows.append({
            "테이블명": table_name,
            "컬럼명": field.name,
            "자료형": field.field_type,
            "모드": field.mode,
        })

schema_df = pd.DataFrame(schema_rows)

with pd.option_context("display.max_rows", None):
    display(schema_df)

,테이블명,컬럼명,자료형,모드
0,accounts_userquestionrecord,id,INTEGER,NULLABLE
1,accounts_userquestionrecord,status,STRING,NULLABLE
2,accounts_userquestionrecord,created_at,TIMESTAMP,NULLABLE
3,accounts_userquestionrecord,chosen_user_id,INTEGER,NULLABLE
4,accounts_userquestionrecord,question_id,INTEGER,NULLABLE
5,accounts_userquestionrecord,user_id,INTEGER,NULLABLE
6,accounts_userquestionrecord,question_piece_id,INTEGER,NULLABLE
7,accounts_userquestionrecord,has_read,INTEGER,NULLABLE
8,accounts_userquestionrecord,answer_status,STRING,NULLABLE
9,accounts_userquestionrecord,answer_updated_at,TIMESTAMP,NULLABLE


In [4]:
import pandas as pd
from google.cloud import bigquery

table_names = [
    "accounts_userquestionrecord",
    "polls_questionset",
    "hackle_properties",
]

# 세 테이블에서 공통으로 확인할 항목
sql = "\nUNION ALL\n".join(
    f"""
    SELECT
        '{name}' AS table_name,
        COUNT(*) AS total_rows,
        COUNTIF(id IS NULL) AS id_nulls,
        COUNT(id) - COUNT(DISTINCT id) AS id_duplicate_extra_rows,
        COUNTIF(user_id IS NULL) AS user_id_nulls,
        COUNTIF(TRIM(CAST(user_id AS STRING)) = '') AS user_id_blanks
    FROM `{PROJECT_ID}.sns_analysis.{name}`
    """
    for name in table_names
)

# 실제 조회 전 예상 처리량 확인
dry_job = client.query(
    sql,
    job_config=bigquery.QueryJobConfig(
        dry_run=True,
        use_query_cache=False,
    ),
)

estimated_bytes = dry_job.total_bytes_processed
max_bytes = 1024**3  # 이번 쿼리의 상한: 1 GiB

if estimated_bytes is None:
    print("예상 처리량을 확인할 수 없어 조회를 실행하지 않았어.")

elif estimated_bytes > max_bytes:
    print(f"예상 처리량: {estimated_bytes / 1024**2:,.2f} MiB")
    print("1 GiB 상한을 초과해서 조회를 실행하지 않았어.")

else:
    print(f"예상 처리량: {estimated_bytes / 1024**2:,.2f} MiB")

    query_job = client.query(
        sql,
        job_config=bigquery.QueryJobConfig(
            maximum_bytes_billed=max_bytes,
        ),
    )

    check_df = pd.DataFrame(
        [dict(row.items()) for row in query_job.result()]
    )
    check_df.columns = [
        "테이블명", "전체행수", "id_결측",
        "id_중복추가행", "user_id_결측", "user_id_공백",
    ]

    display(check_df)

예상 처리량: 31.10 MiB


,테이블명,전체행수,id_결측,id_중복추가행,user_id_결측,user_id_공백
0,accounts_userquestionrecord,1217558,0,0,0,0
1,polls_questionset,158384,0,0,0,0
2,hackle_properties,525350,0,0,0,82255


In [ ]:
def load_table(table_name):
    sql = f"SELECT * FROM `{PROJECT_ID}.sns_analysis.{table_name}`"

    # 실행 전 예상 처리량 확인
    dry_job = client.query(
        sql,
        job_config=bigquery.QueryJobConfig(
            dry_run=True,
            use_query_cache=False,
        ),
    )

    estimated = dry_job.total_bytes_processed

    if estimated is None:
        raise ValueError("예상 처리량을 확인할 수 없어 실행하지 않았어.")

    print(f"{table_name}: 예상 {estimated / 1024**2:,.2f} MiB")

    if estimated > 1024**3:
        raise ValueError("1 GiB 상한을 초과해서 실행하지 않았어.")

    # 전체 데이터를 DataFrame으로 불러오기
    job = client.query(
        sql,
        job_config=bigquery.QueryJobConfig(
            maximum_bytes_billed=1024**3,
        ),
    )

    return job.to_dataframe(create_bqstorage_client=False)


df_record = load_table("accounts_userquestionrecord")
df_questionset = load_table("polls_questionset")
df_hackle = load_table("hackle_properties")

accounts_userquestionrecord: 예상 99.86 MiB
polls_questionset: 예상 20.91 MiB
hackle_properties: 예상 57.97 MiB


In [6]:
for name, df in [
    ("accounts_userquestionrecord", df_record),
    ("polls_questionset", df_questionset),
    ("hackle_properties", df_hackle),
]:
    print(f"\n{name}")

    # 자료형, 컬럼별 값이 있는 행 수
    df.info(show_counts=True)

    # 숫자·문자·날짜 컬럼의 요약 통계
    display(df.describe(include="all").T)


accounts_userquestionrecord
<class 'pandas.DataFrame'>
RangeIndex: 1217558 entries, 0 to 1217557
Data columns (total 12 columns):
 #   Column             Non-Null Count    Dtype              
---  ------             --------------    -----              
 0   id                 1217558 non-null  Int64              
 1   status             1217558 non-null  str                
 2   created_at         1217558 non-null  datetime64[us, UTC]
 3   chosen_user_id     1217558 non-null  Int64              
 4   question_id        1217558 non-null  Int64              
 5   user_id            1217558 non-null  Int64              
 6   question_piece_id  1217558 non-null  Int64              
 7   has_read           1217558 non-null  Int64              
 8   answer_status      1217558 non-null  str                
 9   answer_updated_at  1217558 non-null  datetime64[us, UTC]
 10  report_count       1217558 non-null  Int64              
 11  opened_times       1217558 non-null  Int64              
d

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
id,1217558.0,<NA>,<NA>,<NA>,59572979.018822,771777.0,15539652.75,53026805.0,94809596.25,161666464.0,46081927.886242
status,1217558,3,C,1156322,NaN,NaN,NaN,NaN,NaN,NaN,NaN
created_at,1217558,NaN,NaN,NaN,2023-05-17 12:51:25.947101+00:00,2023-04-28 12:27:49+00:00,2023-05-10 01:42:11+00:00,2023-05-15 15:43:01.500000+00:00,2023-05-22 11:01:03.750000+00:00,2024-05-08 01:36:18+00:00,NaN
chosen_user_id,1217558.0,<NA>,<NA>,<NA>,1092603.584565,833112.0,883692.0,1091749.0,1235698.0,1579422.0,202758.478202
question_id,1217558.0,<NA>,<NA>,<NA>,684.389544,99.0,275.0,469.0,942.0,5133.0,625.039927
user_id,1217558.0,<NA>,<NA>,<NA>,1105789.856924,838023.0,884619.0,1117537.0,1259186.0,1583358.0,206270.380697
question_piece_id,1217558.0,<NA>,<NA>,<NA>,74132114.86932,998458.0,18541420.5,66168425.5,117673456.75,208351468.0,57572369.476189
has_read,1217558.0,<NA>,<NA>,<NA>,0.555153,0.0,0.0,1.0,1.0,1.0,0.496949
answer_status,1217558,3,N,1097932,NaN,NaN,NaN,NaN,NaN,NaN,NaN
answer_updated_at,1217558,NaN,NaN,NaN,2023-05-17 13:44:38.920360+00:00,2023-04-28 12:27:49+00:00,2023-05-10 02:38:44.750000+00:00,2023-05-15 16:29:08.500000+00:00,2023-05-22 11:41:54.750000+00:00,2024-05-08 01:36:18+00:00,NaN



polls_questionset
<class 'pandas.DataFrame'>
RangeIndex: 158384 entries, 0 to 158383
Data columns (total 6 columns):
 #   Column                  Non-Null Count   Dtype              
---  ------                  --------------   -----              
 0   id                      158384 non-null  Int64              
 1   question_piece_id_list  158384 non-null  str                
 2   opening_time            158384 non-null  datetime64[us, UTC]
 3   status                  158384 non-null  str                
 4   created_at              158384 non-null  datetime64[us, UTC]
 5   user_id                 158384 non-null  Int64              
dtypes: Int64(2), datetime64[us, UTC](2), str(2)
memory usage: 23.0 MB


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
id,158384.0,<NA>,<NA>,<NA>,7641273.830696,99817.0,1953393.25,6757335.0,12175103.25,20838446.0,5943689.973322
question_piece_id_list,158384,158384,"[79931958, 79931960, 79931962, 79931966, 79931...",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
opening_time,158384,NaN,NaN,NaN,2023-05-17 13:25:41.180415+00:00,2023-04-28 12:27:22+00:00,2023-05-09 15:46:49.500000+00:00,2023-05-15 13:43:55.500000+00:00,2023-05-22 11:43:24.500000+00:00,2024-05-07 12:12:30+00:00,NaN
status,158384,3,F,153411,NaN,NaN,NaN,NaN,NaN,NaN,NaN
created_at,158384,NaN,NaN,NaN,2023-05-17 12:43:12.170648+00:00,2023-04-28 12:27:23+00:00,2023-05-09 14:58:05.500000+00:00,2023-05-15 13:04:46.500000+00:00,2023-05-22 11:04:03+00:00,2024-05-07 11:32:30+00:00,NaN
user_id,158384.0,<NA>,<NA>,<NA>,1106751.980939,838023.0,884747.0,1117999.0,1261250.0,1583358.0,207861.209058



hackle_properties
<class 'pandas.DataFrame'>
RangeIndex: 525350 entries, 0 to 525349
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   id           525350 non-null  Int64
 1   session_id   525350 non-null  str  
 2   user_id      525350 non-null  str  
 3   language     525350 non-null  str  
 4   osname       525350 non-null  str  
 5   osversion    525350 non-null  str  
 6   versionname  525350 non-null  str  
 7   device_id    525350 non-null  str  
dtypes: Int64(1), str(7)
memory usage: 79.5 MB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,525350.0,<NA>,<NA>,<NA>,262675.5,151655.626297,1.0,131338.25,262675.5,394012.75,525350.0
session_id,525350,253616,vheiXNIAkbRNz8OPRfItfkKxm1A2,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
user_id,525350,327381,,82255,NaN,NaN,NaN,NaN,NaN,NaN,NaN
language,525350,151,ko-KR,340900,NaN,NaN,NaN,NaN,NaN,NaN,NaN
osname,525350,2,iOS,359479,NaN,NaN,NaN,NaN,NaN,NaN,NaN
osversion,525350,74,16.5.1,218699,NaN,NaN,NaN,NaN,NaN,NaN,NaN
versionname,525350,16,2.0.5,309644,NaN,NaN,NaN,NaN,NaN,NaN,NaN
device_id,525350,251720,040914e1-61ac-40ef-b76a-718066d880dc,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
from google.cloud import bigquery
import pandas as pd

# ==========================================
# 기본 설정
# ==========================================

PROJECT_ID = "sns-analysis-prj"
DATASET_ID = "sns_analysis"
LOCATION = "asia-northeast3"

TABLE_NAMES = [
    "accounts_userquestionrecord",
    "polls_questionset",
    "hackle_properties",
]

client = bigquery.Client(
    project=PROJECT_ID,
    location=LOCATION
)

# 테이블 1개당 최대 1 GiB까지만 허용
MAX_BYTES = 1024 ** 3

# False: 예상 연산량만 확인
# True: 실제 점검 쿼리 실행
RUN_QUERY = True

estimated_total_bytes = 0


# ==========================================
# 날짜 순서 점검 조건
# ==========================================

date_order_rules = {
    "accounts_userquestionrecord": [
        (
            "answer_updated_at < created_at",
            """
            `answer_updated_at` IS NOT NULL
            AND `created_at` IS NOT NULL
            AND `answer_updated_at` < `created_at`
            """
        )
    ],

    "polls_questionset": [
        (
            "opening_time < created_at",
            """
            `opening_time` IS NOT NULL
            AND `created_at` IS NOT NULL
            AND `opening_time` < `created_at`
            """
        )
    ],
}


# ==========================================
# 결과 저장 공간
# ==========================================

summary_rows = []
missing_rows = []
date_rows = []


def to_int(value):
    """NULL이나 NaN이면 0으로 변환"""
    return 0 if pd.isna(value) else int(value)


# ==========================================
# 테이블별 점검
# ==========================================

for table_name in TABLE_NAMES:

    table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"

    # 컬럼 정보만 가져오기
    table = client.get_table(table_id)
    fields = list(table.schema)

    column_names = [
        f"`{field.name}`"
        for field in fields
    ]

    group_columns = ", ".join(column_names)

    # 기본 점검 항목
    metrics = [
        """
        COALESCE(
            SUM(row_count),
            0
        ) AS total_rows
        """,

        """
        COUNTIF(
            row_count > 1
        ) AS duplicate_groups
        """,

        """
        COALESCE(
            SUM(
                IF(
                    row_count > 1,
                    row_count - 1,
                    0
                )
            ),
            0
        ) AS duplicate_extra_rows
        """,
    ]

    # --------------------------------------
    # 컬럼별 결측값 점검
    # --------------------------------------

    for field in fields:

        column = f"`{field.name}`"

        # 모든 컬럼의 NULL 개수
        metrics.append(
            f"""
            COALESCE(
                SUM(
                    IF(
                        {column} IS NULL,
                        row_count,
                        0
                    )
                ),
                0
            ) AS `{field.name}__null`
            """
        )

        # 문자열 컬럼의 빈 문자열과 공백
        if field.field_type == "STRING":

            metrics.append(
                f"""
                COALESCE(
                    SUM(
                        IF(
                            {column} IS NOT NULL
                            AND TRIM({column}) = '',
                            row_count,
                            0
                        )
                    ),
                    0
                ) AS `{field.name}__blank`
                """
            )

        # ----------------------------------
        # 날짜 이상 점검
        # ----------------------------------

        if field.field_type == "TIMESTAMP":

            # 2000년 이전 날짜
            metrics.append(
                f"""
                COALESCE(
                    SUM(
                        IF(
                            {column} < TIMESTAMP('2000-01-01'),
                            row_count,
                            0
                        )
                    ),
                    0
                ) AS `{field.name}__before_2000`
                """
            )

            # 미래 날짜
            metrics.append(
                f"""
                COALESCE(
                    SUM(
                        IF(
                            {column} > CURRENT_TIMESTAMP(),
                            row_count,
                            0
                        )
                    ),
                    0
                ) AS `{field.name}__future`
                """
            )

    # --------------------------------------
    # 날짜 순서 점검
    # --------------------------------------

    for check_name, condition in date_order_rules.get(
        table_name,
        []
    ):

        alias = (
            check_name
            .replace(" ", "_")
            .replace("<", "before")
        )

        metrics.append(
            f"""
            COALESCE(
                SUM(
                    IF(
                        {condition},
                        row_count,
                        0
                    )
                ),
                0
            ) AS `{alias}`
            """
        )

    # --------------------------------------
    # 실제 SQL
    # --------------------------------------

    sql = f"""
        WITH grouped AS (

            SELECT
                {group_columns},
                COUNT(*) AS row_count

            FROM `{table_id}`

            GROUP BY
                {group_columns}
        )

        SELECT
            {", ".join(metrics)}

        FROM grouped
    """

    # ======================================
    # Dry Run: 예상 연산량만 확인
    # ======================================

    dry_config = bigquery.QueryJobConfig(
        dry_run=True,
        use_query_cache=False
    )

    dry_job = client.query(
        sql,
        job_config=dry_config
    )

    estimated_bytes = dry_job.total_bytes_processed
    estimated_total_bytes += estimated_bytes

    print(
        f"{table_name} 예상 연산량: "
        f"{estimated_bytes / (1024 ** 2):,.2f} MiB "
        f"({estimated_bytes / (1024 ** 3):.4f} GiB)"
    )

    # 1 GiB 초과 시 무조건 중단
    if estimated_bytes > MAX_BYTES:

        print(
            "→ 1 GiB를 초과해서 "
            "실제 쿼리는 실행하지 않습니다.\n"
        )

        continue

    # 현재는 비용만 확인
    if not RUN_QUERY:

        print(
            "→ Dry Run 완료: "
            "실제 쿼리는 실행하지 않았습니다.\n"
        )

        continue

    # ======================================
    # 실제 점검 쿼리 실행
    # RUN_QUERY = True일 때만 실행됨
    # ======================================

    run_config = bigquery.QueryJobConfig(
        maximum_bytes_billed=MAX_BYTES
    )

    query_job = client.query(
        sql,
        job_config=run_config
    )

    result_df = query_job.to_dataframe()
    result = result_df.iloc[0]

    total_rows = to_int(result["total_rows"])

    # 전체 행 중복 결과
    summary_rows.append({
        "테이블명": table_name,
        "전체행수": total_rows,
        "전체중복그룹": to_int(
            result["duplicate_groups"]
        ),
        "중복추가행": to_int(
            result["duplicate_extra_rows"]
        ),
    })

    # 컬럼별 결측값 결과
    for field in fields:

        null_count = to_int(
            result[f"{field.name}__null"]
        )

        blank_count = 0

        if field.field_type == "STRING":

            blank_count = to_int(
                result[f"{field.name}__blank"]
            )

        missing_count = null_count + blank_count

        missing_rows.append({
            "테이블명": table_name,
            "컬럼명": field.name,
            "자료형": field.field_type,
            "NULL": null_count,
            "빈문자열_공백": blank_count,
            "결측합계": missing_count,
            "결측비율(%)": round(
                missing_count / total_rows * 100,
                4
            ) if total_rows else 0,
        })

        # 날짜 결과
        if field.field_type == "TIMESTAMP":

            date_rows.append({
                "테이블명": table_name,
                "점검내용": f"{field.name} 2000년 이전",
                "문제행수": to_int(
                    result[
                        f"{field.name}__before_2000"
                    ]
                ),
            })

            date_rows.append({
                "테이블명": table_name,
                "점검내용": f"{field.name} 미래 날짜",
                "문제행수": to_int(
                    result[
                        f"{field.name}__future"
                    ]
                ),
            })

    # 날짜 순서 결과
    for check_name, _ in date_order_rules.get(
        table_name,
        []
    ):

        alias = (
            check_name
            .replace(" ", "_")
            .replace("<", "before")
        )

        date_rows.append({
            "테이블명": table_name,
            "점검내용": check_name,
            "문제행수": to_int(
                result[alias]
            ),
        })


# ==========================================
# 예상 연산량 합계
# ==========================================

print(
    "총 예상 연산량: "
    f"{estimated_total_bytes / (1024 ** 2):,.2f} MiB "
    f"({estimated_total_bytes / (1024 ** 3):.4f} GiB)"
)


# ==========================================
# 실제 실행했을 때만 결과 출력
# ==========================================

if RUN_QUERY:

    print("\n1. 전체 행 중복")
    display(pd.DataFrame(summary_rows))

    print("\n2. 컬럼별 결측값")
    display(pd.DataFrame(missing_rows))

    print("\n3. 날짜 점검")
    display(pd.DataFrame(date_rows))

else:

    print(
        "\n현재는 Dry Run만 완료했습니다. "
        "실제 데이터 조회·수정·삭제는 하지 않았습니다."
    )

/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


accounts_userquestionrecord 예상 연산량: 99.86 MiB (0.0975 GiB)


/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


polls_questionset 예상 연산량: 20.91 MiB (0.0204 GiB)


/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


...

,테이블명,전체행수,전체중복그룹,중복추가행
0,accounts_userquestionrecord,1217558,0,0
1,polls_questionset,158384,0,0
2,hackle_properties,525350,0,0



2. 컬럼별 결측값


,테이블명,컬럼명,자료형,NULL,빈문자열_공백,결측합계,결측비율(%)
0,accounts_userquestionrecord,id,INTEGER,0,0,0,0.0000
1,accounts_userquestionrecord,status,STRING,0,0,0,0.0000
2,accounts_userquestionrecord,created_at,TIMESTAMP,0,0,0,0.0000
3,accounts_userquestionrecord,chosen_user_id,INTEGER,0,0,0,0.0000
4,accounts_userquestionrecord,question_id,INTEGER,0,0,0,0.0000
5,accounts_userquestionrecord,user_id,INTEGER,0,0,0,0.0000
6,accounts_userquestionrecord,question_piece_id,INTEGER,0,0,0,0.0000
7,accounts_userquestionrecord,has_read,INTEGER,0,0,0,0.0000
8,accounts_userquestionrecord,answer_status,STRING,0,0,0,0.0000
9,accounts_userquestionrecord,answer_updated_at,TIMESTAMP,0,0,0,0.0000



3. 날짜 점검


,테이블명,점검내용,문제행수
0,accounts_userquestionrecord,created_at 2000년 이전,0
1,accounts_userquestionrecord,created_at 미래 날짜,0
2,accounts_userquestionrecord,answer_updated_at 2000년 이전,0
3,accounts_userquestionrecord,answer_updated_at 미래 날짜,0
4,accounts_userquestionrecord,answer_updated_at < created_at,1426
5,polls_questionset,opening_time 2000년 이전,0
6,polls_questionset,opening_time 미래 날짜,0
7,polls_questionset,created_at 2000년 이전,0
8,polls_questionset,created_at 미래 날짜,0
9,polls_questionset,opening_time < created_at,679


In [27]:
from google.cloud import bigquery
import pandas as pd

MAX_BYTES = 1024 ** 3  # 쿼리당 최대 1 GiB


# 쿼리 1개당 최대 허용량
MAX_BYTES = 1024 ** 3  # 1 GiB

# False: 비용만 확인
# True: 실제 SELECT 실행
RUN_QUERY = True

# 쿼리별 예상 처리량 저장
estimated_queries = {}


def run_select(sql, check_name):

    # 1. Dry Run으로 예상 처리량만 계산
    dry_config = bigquery.QueryJobConfig(
        dry_run=True,
        use_query_cache=False
    )

    dry_job = client.query(
        sql,
        job_config=dry_config
    )

    estimated_bytes = dry_job.total_bytes_processed
    estimated_queries[check_name] = estimated_bytes

    print(
        f"{check_name} 예상 처리량: "
        f"{estimated_bytes / (1024 ** 2):,.2f} MiB "
        f"({estimated_bytes / (1024 ** 3):.4f} GiB)"
    )

    # 2. 제한 초과 시 실행 금지
    if estimated_bytes > MAX_BYTES:
        print("→ 1 GiB를 초과하여 실행하지 않습니다.")
        return None

    # 3. 현재는 Dry Run만 실행
    if not RUN_QUERY:
        print("→ 비용 확인만 완료했습니다.\n")
        return None

    # 4. RUN_QUERY=True일 때만 실제 SELECT 실행
    run_config = bigquery.QueryJobConfig(
        maximum_bytes_billed=MAX_BYTES
    )

    return client.query(
        sql,
        job_config=run_config
    ).to_dataframe()

### 1. 날짜 컬럼의 실제 최솟값·최댓값

In [19]:
sql_date_range = """
WITH date_values AS (

    SELECT
        'accounts_userquestionrecord' AS table_name,
        date_info.column_name,
        date_info.date_value

    FROM `sns-analysis-prj.sns_analysis.accounts_userquestionrecord`

    CROSS JOIN UNNEST([
        STRUCT(
            'created_at' AS column_name,
            created_at AS date_value
        ),
        STRUCT(
            'answer_updated_at' AS column_name,
            answer_updated_at AS date_value
        )
    ]) AS date_info

    UNION ALL

    SELECT
        'polls_questionset' AS table_name,
        date_info.column_name,
        date_info.date_value

    FROM `sns-analysis-prj.sns_analysis.polls_questionset`

    CROSS JOIN UNNEST([
        STRUCT(
            'created_at' AS column_name,
            created_at AS date_value
        ),
        STRUCT(
            'opening_time' AS column_name,
            opening_time AS date_value
        )
    ]) AS date_info
)

SELECT
    table_name,
    column_name,
    MIN(date_value) AS min_date,
    MAX(date_value) AS max_date,
    COUNTIF(date_value IS NULL) AS null_count

FROM date_values

GROUP BY
    table_name,
    column_name

ORDER BY
    table_name,
    column_name
"""

date_range_df = run_select(
    sql_date_range,
    "날짜 최솟값·최댓값 점검"
)

if date_range_df is not None:
    display(date_range_df)

날짜 최솟값·최댓값 점검 예상 처리량: 21.00 MiB (0.0205 GiB)


/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,table_name,column_name,min_date,max_date,null_count
0,accounts_userquestionrecord,answer_updated_at,2023-04-28 12:27:49+00:00,2024-05-08 01:36:18+00:00,0
1,accounts_userquestionrecord,created_at,2023-04-28 12:27:49+00:00,2024-05-08 01:36:18+00:00,0
2,polls_questionset,created_at,2023-04-28 12:27:23+00:00,2024-05-07 11:32:30+00:00,0
3,polls_questionset,opening_time,2023-04-28 12:27:22+00:00,2024-05-07 12:12:30+00:00,0


### 2. 날짜 순서가 뒤집힌 정도 확인

In [20]:
sql_date_reverse = """
SELECT
    'accounts_userquestionrecord' AS table_name,
    'answer_updated_at_before_created_at' AS check_type,

    COUNTIF(
        answer_updated_at < created_at
    ) AS issue_count,

    ROUND(
        SAFE_DIVIDE(
            COUNTIF(answer_updated_at < created_at),
            COUNT(*)
        ) * 100,
        4
    ) AS issue_ratio,

    MIN(
        IF(
            answer_updated_at < created_at,
            TIMESTAMP_DIFF(
                created_at,
                answer_updated_at,
                MILLISECOND
            ) / 1000.0,
            NULL
        )
    ) AS min_gap_seconds,

    APPROX_QUANTILES(
        IF(
            answer_updated_at < created_at,
            TIMESTAMP_DIFF(
                created_at,
                answer_updated_at,
                MILLISECOND
            ) / 1000.0,
            NULL
        ),
        100
    )[OFFSET(50)] AS median_gap_seconds,

    MAX(
        IF(
            answer_updated_at < created_at,
            TIMESTAMP_DIFF(
                created_at,
                answer_updated_at,
                MILLISECOND
            ) / 1000.0,
            NULL
        )
    ) AS max_gap_seconds

FROM `sns-analysis-prj.sns_analysis.accounts_userquestionrecord`

UNION ALL

SELECT
    'polls_questionset' AS table_name,
    'opening_time_before_created_at' AS check_type,

    COUNTIF(
        opening_time < created_at
    ) AS issue_count,

    ROUND(
        SAFE_DIVIDE(
            COUNTIF(opening_time < created_at),
            COUNT(*)
        ) * 100,
        4
    ) AS issue_ratio,

    MIN(
        IF(
            opening_time < created_at,
            TIMESTAMP_DIFF(
                created_at,
                opening_time,
                MILLISECOND
            ) / 1000.0,
            NULL
        )
    ) AS min_gap_seconds,

    APPROX_QUANTILES(
        IF(
            opening_time < created_at,
            TIMESTAMP_DIFF(
                created_at,
                opening_time,
                MILLISECOND
            ) / 1000.0,
            NULL
        ),
        100
    )[OFFSET(50)] AS median_gap_seconds,

    MAX(
        IF(
            opening_time < created_at,
            TIMESTAMP_DIFF(
                created_at,
                opening_time,
                MILLISECOND
            ) / 1000.0,
            NULL
        )
    ) AS max_gap_seconds

FROM `sns-analysis-prj.sns_analysis.polls_questionset`
"""

date_reverse_df = run_select(
    sql_date_reverse,
    "date_reverse_check"
)

if date_reverse_df is not None:
    display(date_reverse_df)

date_reverse_check 예상 처리량: 21.00 MiB (0.0205 GiB)


/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,table_name,check_type,issue_count,issue_ratio,min_gap_seconds,median_gap_seconds,max_gap_seconds
0,polls_questionset,opening_time_before_created_at,679,0.4287,1.0,1.0,29.0
1,accounts_userquestionrecord,answer_updated_at_before_created_at,1426,0.1171,1.0,1.0,1.0


### 3. 동일한 세션·기기에 여러 user_id가 붙었는지 확인

In [21]:
sql_id_mapping = """
WITH cleaned AS (
    SELECT
        TRIM(user_id) AS user_id,
        TRIM(session_id) AS session_id,
        TRIM(device_id) AS device_id

    FROM `sns-analysis-prj.sns_analysis.hackle_properties`

    WHERE user_id IS NOT NULL
      AND TRIM(user_id) != ''
),

mapping_check AS (
    SELECT
        'session_id_to_user_id' AS mapping_type,
        session_id AS key_value,
        COUNT(DISTINCT user_id) AS linked_user_count,
        COUNT(*) AS row_count,
        ARRAY_AGG(
            DISTINCT user_id
            ORDER BY user_id
            LIMIT 10
        ) AS sample_user_ids

    FROM cleaned

    WHERE session_id IS NOT NULL
      AND session_id != ''

    GROUP BY session_id

    HAVING COUNT(DISTINCT user_id) > 1

    UNION ALL

    SELECT
        'device_id_to_user_id' AS mapping_type,
        device_id AS key_value,
        COUNT(DISTINCT user_id) AS linked_user_count,
        COUNT(*) AS row_count,
        ARRAY_AGG(
            DISTINCT user_id
            ORDER BY user_id
            LIMIT 10
        ) AS sample_user_ids

    FROM cleaned

    WHERE device_id IS NOT NULL
      AND device_id != ''

    GROUP BY device_id

    HAVING COUNT(DISTINCT user_id) > 1
),

ranked AS (
    SELECT
        *,
        COUNT(*) OVER (
            PARTITION BY mapping_type
        ) AS total_conflicting_keys,

        ROW_NUMBER() OVER (
            PARTITION BY mapping_type
            ORDER BY linked_user_count DESC, row_count DESC
        ) AS example_rank

    FROM mapping_check
)

SELECT
    mapping_type,
    total_conflicting_keys,
    key_value,
    linked_user_count,
    row_count,
    sample_user_ids

FROM ranked

WHERE example_rank <= 100

ORDER BY
    mapping_type,
    example_rank
"""

id_mapping_df = run_select(
    sql_id_mapping,
    "hackle_id_mapping_check"
)

if id_mapping_df is not None:
    display(id_mapping_df)

hackle_id_mapping_check 예상 처리량: 41.05 MiB (0.0401 GiB)


/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,mapping_type,total_conflicting_keys,key_value,linked_user_count,row_count,sample_user_ids
0,device_id_to_user_id,87962,040914e1-61ac-40ef-b76a-718066d880dc,5,7,"[1577930, 1577938, 1577954, 838541, 849763]"
1,device_id_to_user_id,87962,66A4E7DC-E8B6-4237-9758-4201B0CEB8F5,3,5,"[1411626, 1579612, OarE6jMDaATVnyF9W7nO2PPZIRR2]"
2,device_id_to_user_id,87962,755E96D9-A7A4-4E15-A688-759D7FB53963,3,5,"[1571870, 1572309, ANDeB27PsAXB3bsLyNPb0G0oOQo1]"
3,device_id_to_user_id,87962,B941F9F9-CF53-4DAE-A204-75E666B5D277,3,5,"[1579057, 1579831, 947584]"
4,device_id_to_user_id,87962,80cdff0b-b9c6-414e-b45e-eb3a2e2780bc,3,4,"[1144883, 1425325, 3hOXaWfjz7fFtQbeo6qUaMApIaA3]"
...,...,...,...,...,...,...
195,session_id_to_user_id,87793,cOtgVDbKZSNYW4NIMlpUGw4yIxx1,2,5,"[1240370, cOtgVDbKZSNYW4NIMlpUGw4yIxx1]"
196,session_id_to_user_id,87793,w0eI7Wxp9YPQDnmRDvfKMRemMOh1,2,5,"[1309491, w0eI7Wxp9YPQDnmRDvfKMRemMOh1]"
197,session_id_to_user_id,87793,4xcvMbo4AVZjz7qaZBk3J9gWx6D3,2,5,"[1290083, 4xcvMbo4AVZjz7qaZBk3J9gWx6D3]"
198,session_id_to_user_id,87793,hk9mBAxaNogSkBK81vvM3FNhfSY2,2,5,"[896097, hk9mBAxaNogSkBK81vvM3FNhfSY2]"


In [28]:
sql_identity_match = r"""
SELECT
    COUNT(*) AS total_rows,

    COUNTIF(
        user_id IS NULL OR TRIM(user_id) = ''
    ) AS blank_user_rows,

    COUNTIF(
        REGEXP_CONTAINS(TRIM(user_id), r'^[0-9]+$')
    ) AS numeric_user_rows,

    COUNTIF(
        TRIM(user_id) != ''
        AND NOT REGEXP_CONTAINS(TRIM(user_id), r'^[0-9]+$')
    ) AS string_user_rows,

    COUNTIF(
        TRIM(user_id) != ''
        AND TRIM(user_id) = TRIM(session_id)
    ) AS user_equals_session_rows,

    COUNTIF(
        TRIM(user_id) != ''
        AND TRIM(user_id) = TRIM(device_id)
    ) AS user_equals_device_rows,

    COUNTIF(
        TRIM(session_id) = TRIM(device_id)
    ) AS session_equals_device_rows

FROM `sns-analysis-prj.sns_analysis.hackle_properties`
"""

identity_match_df = run_select(
    sql_identity_match,
    "hackle_identity_match_check"
)

if identity_match_df is not None:
    display(identity_match_df)

hackle_identity_match_check 예상 처리량: 41.05 MiB (0.0401 GiB)


/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,total_rows,blank_user_rows,numeric_user_rows,string_user_rows,user_equals_session_rows,user_equals_device_rows,session_equals_device_rows
0,525350,82255,334091,109004,92565,0,116567


### 4. 범주형 컬럼의 모든 값 확인

In [22]:
sql_category = """
WITH values_long AS (
    SELECT
        'accounts_userquestionrecord' AS table_name,
        value_info.column_name,
        value_info.column_value

    FROM `sns-analysis-prj.sns_analysis.accounts_userquestionrecord`

    CROSS JOIN UNNEST([
        STRUCT(
            'status' AS column_name,
            CAST(status AS STRING) AS column_value
        ),
        STRUCT(
            'answer_status' AS column_name,
            CAST(answer_status AS STRING) AS column_value
        ),
        STRUCT(
            'has_read' AS column_name,
            CAST(has_read AS STRING) AS column_value
        ),
        STRUCT(
            'report_count' AS column_name,
            CAST(report_count AS STRING) AS column_value
        ),
        STRUCT(
            'opened_times' AS column_name,
            CAST(opened_times AS STRING) AS column_value
        )
    ]) AS value_info

    UNION ALL

    SELECT
        'polls_questionset',
        value_info.column_name,
        value_info.column_value

    FROM `sns-analysis-prj.sns_analysis.polls_questionset`

    CROSS JOIN UNNEST([
        STRUCT(
            'status' AS column_name,
            CAST(status AS STRING) AS column_value
        )
    ]) AS value_info

    UNION ALL

    SELECT
        'hackle_properties',
        value_info.column_name,
        value_info.column_value

    FROM `sns-analysis-prj.sns_analysis.hackle_properties`

    CROSS JOIN UNNEST([
        STRUCT(
            'language' AS column_name,
            CAST(language AS STRING) AS column_value
        ),
        STRUCT(
            'osname' AS column_name,
            CAST(osname AS STRING) AS column_value
        ),
        STRUCT(
            'osversion' AS column_name,
            CAST(osversion AS STRING) AS column_value
        ),
        STRUCT(
            'versionname' AS column_name,
            CAST(versionname AS STRING) AS column_value
        )
    ]) AS value_info
),

normalized AS (
    SELECT
        table_name,
        column_name,
        COALESCE(
            NULLIF(TRIM(column_value), ''),
            '<NULL_OR_BLANK>'
        ) AS column_value

    FROM values_long
),

value_counts AS (
    SELECT
        table_name,
        column_name,
        column_value,
        COUNT(*) AS value_count

    FROM normalized

    GROUP BY
        table_name,
        column_name,
        column_value
)

SELECT
    table_name,
    column_name,

    COUNT(*) OVER (
        PARTITION BY table_name, column_name
    ) AS distinct_value_count,

    column_value,
    value_count,

    ROUND(
        SAFE_DIVIDE(
            value_count,
            SUM(value_count) OVER (
                PARTITION BY table_name, column_name
            )
        ) * 100,
        4
    ) AS value_ratio

FROM value_counts

ORDER BY
    table_name,
    column_name,
    value_count DESC
"""

category_df = run_select(
    sql_category,
    "category_value_check"
)

if category_df is not None:
    display(category_df)

category_value_check 예상 처리량: 48.19 MiB (0.0471 GiB)


/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,table_name,column_name,distinct_value_count,column_value,value_count,value_ratio
0,accounts_userquestionrecord,answer_status,3,N,1097932,90.1749
1,accounts_userquestionrecord,answer_status,3,A,111761,9.1791
2,accounts_userquestionrecord,answer_status,3,P,7865,0.6460
3,accounts_userquestionrecord,has_read,2,1,675931,55.5153
4,accounts_userquestionrecord,has_read,2,0,541627,44.4847
...,...,...,...,...,...,...
261,hackle_properties,versionname,16,1.2.9,1,0.0002
262,hackle_properties,versionname,16,2.0.2,1,0.0002
263,polls_questionset,status,3,F,153411,96.8602
264,polls_questionset,status,3,O,4407,2.7825


### 5. Hackle ID 형식·길이 확인

In [23]:
sql_id_format = r"""
WITH id_values AS (
    SELECT
        id_info.column_name,
        TRIM(id_info.id_value) AS id_value

    FROM `sns-analysis-prj.sns_analysis.hackle_properties`

    CROSS JOIN UNNEST([
        STRUCT(
            'user_id' AS column_name,
            user_id AS id_value
        ),
        STRUCT(
            'session_id' AS column_name,
            session_id AS id_value
        ),
        STRUCT(
            'device_id' AS column_name,
            device_id AS id_value
        )
    ]) AS id_info
),

classified AS (
    SELECT
        column_name,
        id_value,
        LENGTH(id_value) AS id_length,

        CASE
            WHEN id_value IS NULL
                THEN 'NULL'

            WHEN id_value = ''
                THEN 'BLANK'

            WHEN REGEXP_CONTAINS(
                id_value,
                r'^[0-9]+$'
            )
                THEN 'NUMERIC'

            WHEN REGEXP_CONTAINS(
                id_value,
                r'^[A-Za-z0-9_-]+$'
            )
                THEN 'ALPHANUMERIC'

            ELSE 'SPECIAL_CHARACTER'
        END AS id_type

    FROM id_values
)

SELECT
    column_name,
    id_type,
    id_length,
    COUNT(*) AS row_count,
    COUNT(DISTINCT id_value) AS distinct_id_count,

    ARRAY_AGG(
        DISTINCT COALESCE(id_value, '<NULL>')
        LIMIT 5
    ) AS sample_values

FROM classified

GROUP BY
    column_name,
    id_type,
    id_length

ORDER BY
    column_name,
    id_type,
    id_length
"""

id_format_df = run_select(
    sql_id_format,
    "hackle_id_format_check"
)

if id_format_df is not None:
    display(id_format_df)

hackle_id_format_check 예상 처리량: 41.05 MiB (0.0401 GiB)


/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,column_name,id_type,id_length,row_count,distinct_id_count,sample_values
0,device_id,ALPHANUMERIC,36,525350,251720,"[aa2f0ec2-6e64-4937-88c0-139dc1e6ba3a, f54427d..."
1,session_id,ALPHANUMERIC,28,408780,195588,"[752SqR0xcyWltcAkQzGs8ZNhvju1, cSj9TmnrJiUgezo..."
2,session_id,ALPHANUMERIC,36,116570,58028,"[aa2f0ec2-6e64-4937-88c0-139dc1e6ba3a, f54427d..."
3,user_id,ALPHANUMERIC,28,109004,96527,"[LNBCrmHJjEcl8kvoMH6MvdsQPuA3, l153ggZquQfop7p..."
4,user_id,BLANK,0,82255,1,[]
5,user_id,NUMERIC,6,53165,36099,"[857358, 842950, 839602, 853568, 863094]"
6,user_id,NUMERIC,7,280926,194754,"[1093663, 1045884, 1525971, 1101460, 1109170]"


### polls_questionset.question_piece_id_list`가 정상적인 리스트 문자열로 구성됐는지 확인

In [32]:
sql_question_piece_list = """
WITH parsed AS (
    SELECT
        id,
        question_piece_id_list,
        SAFE.PARSE_JSON(question_piece_id_list) AS parsed_json

    FROM `sns-analysis-prj.sns_analysis.polls_questionset`
),

checked AS (
    SELECT
        id,
        question_piece_id_list,
        parsed_json,

        question_piece_id_list IS NULL AS is_null,

        question_piece_id_list IS NOT NULL
        AND TRIM(question_piece_id_list) = '' AS is_blank,

        question_piece_id_list IS NOT NULL
        AND TRIM(question_piece_id_list) != ''
        AND parsed_json IS NULL AS is_invalid_json,

        parsed_json IS NOT NULL
        AND JSON_TYPE(parsed_json) != 'array' AS is_not_array,

        CASE
            WHEN JSON_TYPE(parsed_json) = 'array'
            THEN ARRAY_LENGTH(
                JSON_QUERY_ARRAY(parsed_json)
            )
            ELSE NULL
        END AS item_count,

        CASE
            WHEN JSON_TYPE(parsed_json) = 'array'
            THEN EXISTS (
                SELECT 1
                FROM UNNEST(
                    JSON_QUERY_ARRAY(parsed_json)
                ) AS item
                WHERE JSON_TYPE(item) != 'number'
                   OR SAFE_CAST(
                        JSON_VALUE(item) AS INT64
                   ) IS NULL
            )
            ELSE FALSE
        END AS has_non_integer_item,

        CASE
            WHEN JSON_TYPE(parsed_json) = 'array'
            THEN EXISTS (
                SELECT 1
                FROM UNNEST(
                    JSON_QUERY_ARRAY(parsed_json)
                ) AS item
                GROUP BY TO_JSON_STRING(item)
                HAVING COUNT(*) > 1
            )
            ELSE FALSE
        END AS has_duplicate_item

    FROM parsed
)

SELECT
    COUNT(*) AS total_rows,
    COUNTIF(is_null) AS null_count,
    COUNTIF(is_blank) AS blank_count,
    COUNTIF(is_invalid_json) AS invalid_json_count,
    COUNTIF(is_not_array) AS non_array_count,
    COUNTIF(item_count = 0) AS empty_array_count,
    COUNTIF(has_non_integer_item) AS non_integer_item_rows,
    COUNTIF(has_duplicate_item) AS duplicate_item_rows,
    MIN(item_count) AS min_item_count,
    APPROX_QUANTILES(
        item_count,
        100
    )[OFFSET(50)] AS median_item_count,
    MAX(item_count) AS max_item_count

FROM checked
"""

question_piece_list_df = run_select(
    sql_question_piece_list,
    "question_piece_list_check"
)

if question_piece_list_df is not None:
    display(question_piece_list_df)

question_piece_list_check 예상 처리량: 15.62 MiB (0.0153 GiB)


/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,total_rows,null_count,blank_count,invalid_json_count,non_array_count,empty_array_count,non_integer_item_rows,duplicate_item_rows,min_item_count,median_item_count,max_item_count
0,158384,0,0,0,0,0,0,0,10,10,10


# 1차 데이터 전처리 점검

## 1. 점검 대상

- `accounts_userquestionrecord`
- `polls_questionset`
- `hackle_properties`

## 2. 점검 항목

- 테이블별 전체 행 수와 컬럼 자료형
- 컬럼별 NULL·빈 문자열·공백
- `id` 중복 및 전체 행 중복
- 숫자형 컬럼의 범위
- 날짜 최솟값·최댓값 및 시간 순서
- 범주형 컬럼의 고유값과 빈도
- Hackle ID의 형식과 연결 관계
- `question_piece_id_list`의 문자열 구조

## 3. 점검 결과

### 3-1. 전체 행 수 및 중복

| 테이블 | 전체 행 수 | `id` 결측 | `id` 중복 | 전체 행 중복 |
|---|---:|---:|---:|---:|
| `accounts_userquestionrecord` | 1,217,558 | 0 | 0 | 0 |
| `polls_questionset` | 158,384 | 0 | 0 | 0 |
| `hackle_properties` | 525,350 | 0 | 0 | 0 |

세 테이블 모두 `id` 결측 및 중복이 없었으며, 모든 컬럼 값이 동일한 전체 행 중복도 발견되지 않았다.

### 3-2. 컬럼별 결측값

- `accounts_userquestionrecord`: 모든 컬럼의 NULL·빈 문자열 0건
- `polls_questionset`: 모든 컬럼의 NULL·빈 문자열 0건
- `hackle_properties`: `user_id` 빈 문자열 82,255건
- `hackle_properties.user_id`의 빈 문자열 비율은 15.6572%이다.
- 그 외 Hackle 컬럼에서는 NULL·빈 문자열이 발견되지 않았다.

빈 `user_id`는 회원 ID를 기준으로 한 사용자 추적과 회원 테이블 연결이 어려우므로 분석 대상에서 제외하기로 합의했다. 실제 데이터에 반영하기 전 팀원들과 처리 방식을 다시 확인할 예정이다.

### 3-3. 숫자형 컬럼

- 숫자형 컬럼은 `describe()`를 통해 분포와 최솟값·최댓값을 확인했다.
- `has_read`는 `0`, `1`로 구성된 이진값이다.
- `opened_times`는 0~3 범위이며 초성 힌트를 열어본 횟수로 추정된다.
- `report_count`는 0~14 범위로 확인됐다.
- 명확하게 잘못된 숫자값이 발견되지 않아 별도의 이상치 처리를 하지 않는다.

### 3-4. 날짜 범위

| 테이블 | 컬럼 | 최소 날짜 | 최대 날짜 |
|---|---|---|---|
| `accounts_userquestionrecord` | `created_at` | 2023-04-28 12:27:49 UTC | 2024-05-08 01:36:18 UTC |
| `accounts_userquestionrecord` | `answer_updated_at` | 2023-04-28 12:27:49 UTC | 2024-05-08 01:36:18 UTC |
| `polls_questionset` | `created_at` | 2023-04-28 12:27:23 UTC | 2024-05-07 11:32:30 UTC |
| `polls_questionset` | `opening_time` | 2023-04-28 12:27:22 UTC | 2024-05-07 12:12:30 UTC |

- 날짜 컬럼의 결측값은 모두 0건이다.
- 2000년 이전 날짜와 미래 날짜는 발견되지 않았다.
- 전체 날짜 범위에서 비상식적인 값은 발견되지 않았다.

### 3-5. 날짜 순서 점검

| 점검 조건 | 행 수 | 비율 | 최소 차이 | 중앙값 | 최대 차이 |
|---|---:|---:|---:|---:|---:|
| `answer_updated_at < created_at` | 1,426 | 0.1171% | 1초 | 1초 | 1초 |
| `opening_time < created_at` | 679 | 0.4287% | 1초 | 1초 | 29초 |

일부 행에서 시간 순서가 뒤집혀 있었지만 차이가 대부분 1초이고 최대 29초에 불과했다. 서버 기록 순서 또는 데이터 처리 시점의 미세한 차이로 판단하여 삭제하거나 수정하지 않는다.

### 3-6. 범주형 컬럼

- `answer_status`는 `N`, `A`, `P`의 3개 값으로 구성됐다.
- `has_read`는 `0`, `1`의 2개 값으로 구성됐다.
- `polls_questionset.status`는 `F`, `O`, `C`의 3개 값으로 구성됐다.
- 상태 코드의 정확한 의미는 데이터 정의 확인이 필요하다.
- Hackle의 언어, 운영체제 및 앱 버전값은 실제 사용자 환경에 따라 다양하게 나타날 수 있다.
- 사용 빈도가 낮은 앱 버전도 과거 버전 사용 기록일 수 있으므로 희귀값이라는 이유만으로 삭제하지 않는다.

### 3-7. Hackle ID 형식

| `user_id` 형식 | 행 수 | 고유 ID 수 |
|---|---:|---:|
| 빈 문자열 | 82,255 | 1 |
| 숫자형 6자리 | 53,165 | 36,099 |
| 숫자형 7자리 | 280,926 | 194,754 |
| 영문·숫자형 28자리 | 109,004 | 96,527 |

추가적인 ID 연결 결과는 다음과 같다.

- `user_id = session_id`: 92,565행
- `session_id = device_id`: 116,567행
- `user_id = device_id`: 0행
- 여러 `user_id`가 연결된 `session_id`: 87,793개
- 여러 `user_id`가 연결된 `device_id`: 87,962개

Hackle의 `user_id`에는 숫자형 회원 ID와 영문·숫자형 식별자가 함께 존재한다. 로그인 전에는 세션 또는 익명 식별자를 사용하고, 로그인 후에는 숫자형 회원 ID를 사용하는 구조일 가능성이 있다.

따라서 숫자형 ID와 영문·숫자형 ID는 모두 유지하며, 임의로 통합하거나 삭제하지 않는다.

### 3-8. `question_piece_id_list` 구조

| 점검 항목 | 결과 |
|---|---:|
| 전체 행 수 | 158,384 |
| NULL | 0 |
| 빈 문자열 | 0 |
| JSON 형식 오류 | 0 |
| 리스트가 아닌 값 | 0 |
| 빈 리스트 | 0 |
| 숫자가 아닌 값 포함 | 0 |
| 리스트 내부 중복 ID | 0 |
| 리스트 길이 최솟값 | 10 |
| 리스트 길이 중앙값 | 10 |
| 리스트 길이 최댓값 | 10 |

`question_piece_id_list`는 전체 행에서 정상적인 리스트 형식으로 저장돼 있었다. 모든 리스트가 숫자형 `question_piece_id` 10개로 구성됐으며, 리스트 내부 중복이나 형식 오류도 발견되지 않았다.

따라서 해당 컬럼은 별도로 처리하지 않고 원본 그대로 사용한다.

## 4. 전처리 결정

- 전체 행 중복과 `id` 중복이 없으므로 중복 제거를 하지 않는다.
- 날짜 범위에는 이상이 없으며, 미세한 시간 역전도 원본 그대로 유지한다.
- 숫자형 컬럼에서 명확한 오류가 발견되지 않아 별도의 이상치 처리를 하지 않는다.
- 범주형 컬럼의 희귀값은 실제 상태 및 사용환경일 수 있으므로 임의로 변경하지 않는다.
- `question_piece_id_list`는 정상적인 구조이므로 그대로 사용한다.
- Hackle의 숫자형·문자형 ID는 서로 다른 식별 방식일 수 있으므로 모두 유지한다.
- `hackle_properties.user_id`가 빈 문자열인 82,255건만 회원 단위 분석에서 제외할 예정이다.
- 빈 `user_id`의 실제 처리 방식은 팀원들과 최종 확인 후 결정한다.
- 현재까지 원본 데이터를 변경하는 DML 쿼리는 실행하지 않았다.

## 5. 최종 결론

세 테이블의 중복, 결측, 숫자 범위, 날짜, 범주값 및 컬럼 구조를 점검했다. 명확한 데이터 오류는 발견되지 않았으며, 별도 처리 대상은 `hackle_properties.user_id`가 빈 문자열인 82,255건이다.

해당 행의 처리 방식을 팀에서 최종 결정한 후 1차 전처리를 완료한다.